<img src="https://cmse.msu.edu/sites/_cmse/assets/Image/image001.png"
     alt="CMSE Grapical Image"
     style="float: right; margin-right: 10px;" 
     height="164" 
     width="164" />
# __CMSE  201 - Fall 2019__
    

# Homework 6: Practicing all of your data fitting skills

## Goals

### By the end of the homework assignment you will have practiced:

1. Fitting the properties of unfamiliar data using:
    - `polyfit()`
    - `curve_fit()`
    - The Metropolis-Hastings MCMC algorithm
2. Visualizing your results

## Assignment instructions

Work through the following assignment, making sure to follow all of the directions and answer all of the questions.

**This assignment is due at 11:59pm on Friday, November 22** It should be uploaded into the "Homework Assignments" submission folder for Homework #6.  Submission instructions can be found at the end of the notebook.

## Grading

* Question 1 (**2 points**)
* Question 2 (**4 points**)
* Question 3 (**2 points**)
* Question 4 (**8 points**)
* Question 5 (**4 points**)
* Question 6 (**4 points**)
* Question 7 (**8 points**)
* Question 8 (**2 points**)
* Question 9 (**1 point**)
* Question 10 (**1 point**)
* Question 11 (**12 points**)
    
**Total**: 48 points

---
# Becoming a data fitting wizard

At this point in the semester, you've had the opportunity to test your ability to fit models to data and extract best fit parameters. For this assignment, you're going to practice these skills again! 

This time, you're going to be working with a top secret dataset and the origins of the values are classified. While this makes it a bit challenging to know exactly what the right model is, you're going to do your best and report back to the authorities that provided you with the data.

So, **let's get to it!**

The data file that you're going to be working is aptly named `mystery.csv` and you should have downloaded it from D2L along with this notebook. Again, since you don't have any idea exactly what this data represents, you're going to explore various models that might serve as good representations of the general trends within the data.

In [ ]:
###ANSWER
# This code is for generating the data that they'll fit
# MAKE SURE YOU DON'T OVERWRITE THE ORIGINAL DATA THAT IS SHARED WITH THE STUDENTS

import numpy as np
import matplotlib.pyplot as plt

def parabola_wiggles(x, A, B, C, D):
    return (A*x**2 + B*np.sin(C*x) + D) + np.random.normal(-0.7,0.7,size=(len(x)))

x = np.random.uniform(-4,4,size=(300))
y = parabola_wiggles(x, 0.5, 2, 4, -3)
plt.scatter(x,y)

fake_data = np.zeros((len(x),2))
fake_data[:,0] = x
fake_data[:,1] = y
#np.savetxt("mystery.csv",fake_data,delimiter=",",fmt="%.6f")

First things first -- time to load in the data! 

**Question 1 (2 points)**: Using either Pandas or NumPy, read the two columns in the data file into your python notebook and store the the first column as `x` and the second column as `y`.

In [ ]:
# Put your code here


In [ ]:
### ANSWER
import numpy as np
x,y = np.loadtxt("mystery.csv",delimiter=",",unpack=True)

**Question 2 (4 points)**: Now that you've loaded in the data, make a plot that shows how `y` changes as a function of `x`. You should make sure to format your plot so that it's easy to see the relationship between `x` abd `y`. You should also include useful things like axis labels! (Also, feel free to use seaborne if you want to change the overall look of your plot).

In [ ]:
# Put your code here


In [ ]:
###ANSWER
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_style("whitegrid")
sns.set_context("talk")
plt.scatter(x,y)
plt.xlabel("x")
plt.ylabel("y")

**Question 3 (2 points)**: Before you start trying to fit any sort of models, what features do you see in the data? Describe any patterns or large scale behavior you notice in the data.

<font size=20>&#9998;</font> *Put your answer here.*

###ANSWER
They should ideally comment on the large-scale parabola and the small scale oscillations.

**Question 4 (8 points)**: Clearly a line doesn't seem like a good choice to model the properties of this dataset. However, a second order polynomial might be a good fit!

Now do the following:

1. Use NumPy's `polyfit` function to fit a polynomial of degree 2 to the data.
2. Use `poly1d` to create a function that can be used to plot the best fit curve on top of the data.
3. Generate 200 evenly spaced values between -4 and 4 and store those in a new variable called `x_model`
4. Plot the data and then overplot the `y_model` values that correspond to the `x_model` values you just created. You should be able to use the function you created using `poly1d` to do this!

In [ ]:
# Put your code here


In [ ]:
###ANSWER
p = np.polyfit(x,y,2)
poly2 = np.poly1d(p)

x_model = np.linspace(-4,4,200)

plt.scatter(x,y)
plt.plot(x_model,poly2(x_model),color="orange",lw=4)
plt.xlabel("x")
plt.ylabel("y")

**Question 5 (4 points)**: How well does this model appear to fit the data? What are the best fit parameters for this model? **Make sure to print them out**. Are there features that still aren't captured by this model? If so, comment on which features these are.

<font size=20>&#9998;</font> *Put your answer here.*

In [ ]:
###ANSWER 
print("Parameters",p)

###ANSWER The large features are well fit, the periodic information isn't captured.

**Question 6 (4 points)**: Would a higher order polynomial fit the data better? Try fitting higher order polynomials and decide which one you think provides the "best" overall fit to the data. **Defend your choice below!**

*Note*: you may have to experiment a higher order polynomial than you might have ever previously tried! This is where `poly1d()` really comes in handy!

In [ ]:
# Put your code here


<font size=20>&#9998;</font> *Defend your choice for the "best" polynomial that fits the data here!*

In [ ]:
###ANSWER
p = np.polyfit(x,y,13)
polyguess = np.poly1d(p)

x_model = np.linspace(-4,4,200)

plt.scatter(x,y)
plt.plot(x_model,polyguess(x_model),color="orange",lw=4)
plt.xlabel("x")
plt.ylabel("y")

###ANSWER order 13 and order 14 both seem to fit comparably well, bumping to 15 starts to capture smaller scale features that aren't really in the data. That said, if they fit something in this range and can reasonably defend their choice, they should get their points.

**Question 7 (8 points)**: Although you might have been able to get a decent fit to the data using a higher order polynomial, it seem like such a polynomial is an overly complex way of fitting the data. Instead of just experimenting with `polyfit`, it might make more sense to use SciPy's `curve_fit` to fit a simple model to the data.

As you've hopefully noticed at this point, the second order polynomial seemed to do a good job of fitting the large scale behavior of the data, but there also appears to be a periodic signal in the data as well.

**Define a new function that combines a second order polynomial and a sinusoidal function and use `curve_fit` to find the best fit parameters for this new function.**

*Hint*: You may need to experiment with the `p0` parameters that you can feed into `curve_fit` to get a model that captures in the "wiggles" in the dataset.

In [ ]:
# An important import command!
from scipy.optimize import curve_fit

# Put your code here


In [ ]:
###ANSWER
def parabola_wiggles_fit(x, A, B, C, D):
    # Note, they may also end up including an "x" term, this is just the case where that x term has a coefficient of 0
    return (A*x**2 + B*np.sin(C*x) + D)

popt,pcov = curve_fit(parabola_wiggles_fit, x, y, p0=[1, 1, 4, 1])
print(popt)

plt.scatter(x,y)
plt.plot(x_model,parabola_wiggles_fit(x_model,popt[0],popt[1],popt[2],popt[3]),color="orange")

**Question 8 (2 points)**: Were you able to get a better looking fit for the model? How do your best fit parameters from `curve_fit` compare to the best fit parameters from your second order polynomial fit? Are any of them roughly the same? If so, discuss whether or not this what you would expect.

<font size=20>&#9998;</font> *Put your answer here.*

###ANSWER Ideally they will notice that whatever parameter they used for the scaling the parabola will be basically the same, as will the y-intercept value. Which makes sense given the nature of the problem!

---
### Pushing further with MCMC

At this point, you feel like you've done a good job of fitting the data, but you'd really like to make sure you've got the best possible fit for some of your parameters. To do this, you're going to use a Markov Chain Monte Carlo (MCMC) approach. Although you could try using MCMC to calculate best fit values for _all_ of your parameters, you're going to limit your search to **just two** of the parameters in your model to keep things simple.

**Question 9 (1 point)**: Using your model from Question 7, you're going to pick **two** of the model parameters to explore with an MCMC approach and you will *leave all other parameters fixed* based on your results from Question 7. **State which two of your parameters you will be trying to fit with MCMC:**

<font size=20>&#9998;</font> *Which two parameters are you going to try and fit? Answer that here.*

###ANSWER It doesn't matter what they choose as long as it is clear and aligns with what they do in the following questions. For these solutions, we're going to choose the value of B and C from our model above.

**Question 10 (1 point)**: Now, in order to use $\chi^2$ as our "cost function" for computing the goodness of fit, we also need to have an estimate of the error bars on our data points. Since we don't exactly know where the data came from, or what the "right" error bars should be, let's assume that the error for each point is 10% of its value. **Define a new array, `sigma` that is equal to 10% of the original `y` array in the dataset.** 

In [ ]:
# Put your code here


In [ ]:
### ANSWER
sigma = 0.10 * y

**Question 11 (12 points)**: At this point we need to implement the MCMC algorithm to explore parameter space for the **two free parameters** that you chose in Question 9.

**All of your other parameters should be set to constants and you should use the values that came out of your best fit from Question 7**. For example, if you had a parameter `A` and the best fit value was 8.25 then you should set `A = 8.25` and use that in your model when doing the MCMC search for finding the best fit values for your two free parameters.

As a reminder, the equation for $\chi^2$ error is like so:

$$ \chi_R^2 = \frac{1}{N_{pts}}\sum_i \frac{(y_{data,i} - y_{model,i})^2}{2 \sigma_i^2} $$

The function for calculating this error is provided for you in the cell below (and cleverly called `calculate_error`)

#### Implementing the MCMC algorithm 

Using the Day 20 in-class assignment as a guide, you should now try to implement the MCMC algorithm to find the best fit for your two free parameters (part of the Day 20 notebook is also included for reference at the end of this notebook).

You should take **$n = 100000$** steps and use a step size of **0.05**. Remember, you can use your function from Question 7 to compute your model values!

**Upon completing the search, you should make the following plots**:

1. A plot of where your random walker "walked" in parameter space while it tried to find the best fit values (you can try starting your walker whever you want, but if you get strange results, it might be because you started the walker too far away). Your plot should basically be of "parameter 2" vs "parameter 1" as the walker walked for whatever your two parameters are.
1. A 2D histogram and contour plot that highlights where the best fit parameters are.

Finally, **you should print the best fit values and comment on how they compare to the values you found using `curve_fit`.**


In [ ]:
def calculate_error(ys_actual, ys_model, sigma):
    """
    Calculate the chi-squared error between two sets of data
    """
    return ((ys_actual-ys_model)**2/(2*sigma**2)).sum()/(ys_actual.size)

In [ ]:
# Put all of the code necessary for implementing your MCMC search for the best fit parameters and visualizing the results.
# You may wish to create additional cells as necessary


<font size=20>&#9998;</font> What are you best fit parameters? How do these compare to the values that `curve_fit` found?

*Put your answer here.*

In [ ]:
### ANSWER

# Free parameters with random initial guesses
Bs = [3]
Cs = [4]

# Fixed parameters
A = popt[0]
D = popt[3]

# Total number of points we're going to sample (start out with at least 10^4)
num_sample_points = 100000

# Weight factor in front of the random step
step_size = 0.05

# Set up the initial guess values and error
tmp_ys = parabola_wiggles_fit(x, A, Bs[0], Cs[0], D)
chi_squared = calculate_error(y, tmp_ys, sigma)

for i in range(num_sample_points):
    
    # Guess a new set of parabola parameters
    new_B = Bs[-1] + np.random.normal() * step_size
    new_C = Cs[-1] + np.random.normal() * step_size

    # Calculate the RMS error for the parabola defined by the new parameters
    tmp_ys = parabola_wiggles_fit(x, A, new_B, new_C, D)
    chi_squared_tmp = calculate_error(y, tmp_ys, sigma)
    
    # acceptance probability - ratio of likelihoods for old and new data points.
    prob = np.exp(-chi_squared_tmp + chi_squared)
    
    if np.random.uniform() < prob:
        Bs.append(new_B)
        Cs.append(new_C)
        chi_squared = chi_squared_tmp

In [ ]:
### ANSWER
plt.plot(Bs, Cs)

plt.annotate('start', xy=(Bs[0], Cs[0]))
plt.annotate('stop', xy=(Bs[-1], Cs[-1]))

plt.xlabel('Width parameter')
plt.ylabel('Intercept parameter')
plt.title('Markov Chain for estimate of optimal B and C parameters');

In [ ]:
### ANSWER
from matplotlib.colors import LogNorm

fig, [colored, bw] = plt.subplots(1,2, sharey=True, figsize = (16, 5))

counts, xbin, ybin, img = colored.hist2d(Bs, Cs, bins=64, norm=LogNorm())

# use np.argwhere() to find the bin(s) with the max counts
max_location = np.argwhere(counts == counts.max())

# Use the location of the max to find the best width and intercept parameters
best_B = xbin[max_location[0,0]]
best_C = ybin[max_location[0,1]]
colored.annotate('best', xy=(best_B, best_C))

bw.hist2d(Bs, Cs, bins=60, norm=LogNorm(), cmap='gray')
bw.contour(0.5*(xbin[1:]+xbin[:-1]), 0.5*(ybin[1:]+ybin[1:]), counts.transpose(), linewidths=2)

colored.set_xlabel("B parameter")
colored.set_ylabel("C parameter")
bw.set_xlabel("B parameter")

print(best_B, best_C)

###ANSWER They should find that it wanders it way into the same best fit parameters when comparing it to the previous results.

---
## Assignment wrap-up

Please fill out the form that appears when you run the code below.  

In [ ]:
from IPython.display import HTML
HTML(
"""
<iframe 
	src="https://forms.gle/zw3MaBYJWhY9XHrA7" 
	width="800px" 
	height="600px" 
	frameborder="0" 
	marginheight="0" 
	marginwidth="0">
	Loading...
</iframe>
"""
)

---

### Congratulations, you're done!

Submit this assignment by uploading it to the course Desire2Learn web page.  Go to the "Homework Assignments" folder, find the submission link for Homework #6, and upload it there.

---
---
---


# Markov Chain Monte Carlo fitting

## How it works

The general idea behind MCMC fitting of our data is that we will start from a guess at our model parameters and "walk" in random directions in parameter space in a way that **on average gets us closer to the best fit to the data.**  We keep track of the points that we’ve sampled over time (we call this the "trace" of the data), and use those to create a distribution.  The distribution shows us how likely each set of model parameters is to fit the data.

We’re going to think about how this works using the model from our pre-class assignment as an example.
Specifically, you’re going to revisit fitting data of the form

$$
   f(x) = W x^2 + I
$$

by intelligently searching for optimal $W$ and $I$ (_width_ and _intercept_ of the parabola) values. Specifically, Metropolis-Hastings fitting consists of these steps:

1. Start with an initial guess of the model parameters, $(W_0, I_0)$.
2. Calculate $\chi_0^2$ for this initial guess.

Then, the following occurs in a loop over specified number of steps:

3. Take a (potential) "step" from $(W_0, I_0)$ in a random direction to produce $(W_1, I_1)$
4. Calculate the the ["reduced chi-squared"](https://en.wikipedia.org/wiki/Reduced_chi-squared_statistic) values (as done in the pre-class), $\chi_0^2$ and $\chi_1^2$, for the parabolas defined by $(W_0, I_0)$ and $(W_1, I_1)$.
5. Calculate an acceptance probability, $P = e^{-\chi_1^2}/e^{-\chi_0^2}$, as the ratio of two *likelihood functions* (the exponentials).
6. Uniformly generate a random number $r \in [0, 1)$. If $r < P$, "accept" $(W_1, I_1)$ as the next initial guess and assign $\chi_0^2$=$\chi_1^2$. Otherwise, discard $(W_1, I_1)$ and generate a new potential step from $(W_0, I_0)$.
7. Repeat this process until you’ve generated as many points as you care to (100000 isn’t bad).

### Notes:

* The Markov-chain part of Markov-chain Monte Carlo means "the next step only depends on the current step."
* If $\chi_1^2 < \chi_0^2$ (i.e. the error from $(W_1, I_1)$ is less than the error from $(W_0, I_0)$), then $P > 1$ and the new point is _always_ accepted.
* By keeping track of the valid steps, we can chart the progress of a "walker" as it (hopefully!) moves towards a set of optimum values. The walker will tend to stay in the region of good fit but its wandering will inform on the range of likely values.
* The randomness here _usually_ prevents walkers from moving in sub-optimal (higher-error) directions but occasionally allows it to happen in hopes of finding even lower error zones.